# Scenario summary

Numbers only. Figures live in `../plots/`.

All sixteen runs — four GCAM ambition scenarios (SSP-RCP pairs: SSP1-1.9, SSP1-2.6, SSP2-3.7,
SSP2-4.5) crossed with four captive-power CCS regimes — are on MPP's full 132-pair switch table.
Each ambition scenario supplies its own GCAM demand, grid and carbon budget. Sections 23 and 24 of
`MODEL_REFERENCE.md` record why the full switch table is used.

**Smelter-focused.** Refining is inert to the grid and CCS levers and was not re-run on GCAM
demand (the single `REFINERY_REF` run is on the old ~65 Mt demand), so combined smelter+refining
totals would mix demand bases. These tables report smelting only. Interim stand-in GCAM runs.

In [1]:
import sys
import importlib
sys.path.append("../..")

import common
importlib.reload(common)   # pick up edits to common.py without restarting the kernel

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import (SCENARIOS, LABELS, COLOURS, PLANTS, ALUMINA_PER_ALUMINIUM,
                    style, emissions, production, production_by_technology, budget,
                    intensity_split, process_emission_factors, anode, power_source,
                    overlaid, panels, year_axis, save)

style()
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Cumulative smelting emissions against each scenario's GCAM budget

The gap between a cell's cumulative smelting emissions and its own GCAM budget is the asset-resolution signal: where MPP's captive fossil, anode lag and CCS limits break the budget GCAM's smooth pathway implies.

In [2]:
rows = []
for s in SCENARIOS:
    sm = emissions(s, "smelter").sum() / 1000            # Gt CO2, cumulative 2020-2050
    bud = budget("smelter", s).sum() / 1000              # Gt CO2, this scenario's GCAM budget
    rows.append({"Scenario": LABELS[s], "Smelting Gt": sm, "GCAM budget Gt": bud,
                 "vs budget %": 100 * (sm / bud - 1)})
pd.DataFrame(rows).set_index("Scenario").round(2)

,Smelting Gt,GCAM budget Gt,vs budget %
Scenario,,,
"SSP1-1.9, none CCS",10.52,10.24,2.70
"SSP1-1.9, low CCS",10.55,10.24,2.98
"SSP1-1.9, high CCS",10.48,10.24,2.35
"SSP1-1.9, unlimited CCS",10.41,10.24,1.63
"SSP1-2.6, none CCS",12.95,13.03,-0.62
"SSP1-2.6, low CCS",13.08,13.03,0.44
"SSP1-2.6, high CCS",13.19,13.03,1.27
"SSP1-2.6, unlimited CCS",13.14,13.03,0.90
"SSP2-3.7, none CCS",18.07,17.87,1.11


## Emissions intensity, process against electricity (smelter only)

The split is explained in `../plots/03_emissions_intensity.ipynb`. Process here is the smelter anode term only (refining excluded — not re-run on GCAM demand).

In [3]:
factors = process_emission_factors()
years = [2025, 2030, 2035, 2040, 2045, 2050]

def smelter_split(s):
    smelt = production_by_technology(s, "smelter")
    al = smelt.sum(axis=1)
    proc = sum(smelt[t] * factors[anode(t)] for t in smelt.columns)   # anode process + PFC
    tot = emissions(s, "smelter")
    return pd.DataFrame({"Process": proc / al,
                         "Electricity": (tot - proc) / al}).assign(Total=lambda d: d["Process"] + d["Electricity"])

split = {s: smelter_split(s) for s in SCENARIOS}
for component in ["Process", "Electricity", "Total"]:
    print(f"{component} intensity (smelter only), tCO2e per tonne aluminium")
    display(pd.DataFrame({LABELS[s]: split[s][component] for s in SCENARIOS}).loc[years].round(2))

Process intensity (smelter only), tCO2e per tonne aluminium


,"SSP1-1.9, none CCS","SSP1-1.9, low CCS","SSP1-1.9, high CCS","SSP1-1.9, unlimited CCS","SSP1-2.6, none CCS","SSP1-2.6, low CCS","SSP1-2.6, high CCS","SSP1-2.6, unlimited CCS","SSP2-3.7, none CCS","SSP2-3.7, low CCS","SSP2-3.7, high CCS","SSP2-3.7, unlimited CCS","SSP2-4.5, none CCS","SSP2-4.5, low CCS","SSP2-4.5, high CCS","SSP2-4.5, unlimited CCS"
year,,,,,,,,,,,,,,,,
2025,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09
2030,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09,2.09
2035,1.85,1.84,1.84,1.76,1.54,1.49,1.56,1.67,1.87,1.87,1.88,1.87,1.85,1.85,1.87,1.88
2040,1.31,1.32,1.24,1.17,1.19,1.26,1.18,1.37,1.33,1.37,1.36,1.50,1.49,1.42,1.47,1.54
2045,0.94,0.94,0.81,0.72,1.11,1.19,1.10,1.26,1.03,1.03,1.14,1.31,1.29,1.21,1.32,1.46
2050,0.65,0.66,0.57,0.44,1.06,1.13,1.03,1.15,0.99,0.99,0.99,1.20,1.07,1.02,1.19,1.35


Electricity intensity (smelter only), tCO2e per tonne aluminium


,"SSP1-1.9, none CCS","SSP1-1.9, low CCS","SSP1-1.9, high CCS","SSP1-1.9, unlimited CCS","SSP1-2.6, none CCS","SSP1-2.6, low CCS","SSP1-2.6, high CCS","SSP1-2.6, unlimited CCS","SSP2-3.7, none CCS","SSP2-3.7, low CCS","SSP2-3.7, high CCS","SSP2-3.7, unlimited CCS","SSP2-4.5, none CCS","SSP2-4.5, low CCS","SSP2-4.5, high CCS","SSP2-4.5, unlimited CCS"
year,,,,,,,,,,,,,,,,
2025,7.06,7.19,7.20,7.27,6.56,6.32,6.63,6.34,7.13,6.79,7.00,6.98,6.94,6.96,6.98,6.68
2030,0.63,0.62,0.58,0.64,3.51,3.53,3.54,3.44,5.39,5.43,5.36,5.27,5.42,5.38,5.44,5.20
2035,0.51,0.50,0.51,0.61,1.48,1.23,1.60,1.43,4.92,4.85,4.86,4.85,4.87,4.84,4.87,4.84
2040,0.49,0.49,0.54,0.64,0.97,1.30,1.14,1.19,3.80,3.71,3.79,3.52,4.33,4.41,4.37,4.20
2045,0.32,0.32,0.44,0.54,0.82,0.94,1.04,0.93,2.16,2.18,2.09,1.84,3.51,3.58,3.46,3.29
2050,0.29,0.28,0.38,0.51,0.75,0.72,0.79,0.69,1.52,1.54,1.56,1.32,2.65,2.80,2.76,2.51


Total intensity (smelter only), tCO2e per tonne aluminium


,"SSP1-1.9, none CCS","SSP1-1.9, low CCS","SSP1-1.9, high CCS","SSP1-1.9, unlimited CCS","SSP1-2.6, none CCS","SSP1-2.6, low CCS","SSP1-2.6, high CCS","SSP1-2.6, unlimited CCS","SSP2-3.7, none CCS","SSP2-3.7, low CCS","SSP2-3.7, high CCS","SSP2-3.7, unlimited CCS","SSP2-4.5, none CCS","SSP2-4.5, low CCS","SSP2-4.5, high CCS","SSP2-4.5, unlimited CCS"
year,,,,,,,,,,,,,,,,
2025,9.16,9.28,9.30,9.36,8.65,8.42,8.72,8.43,9.22,8.88,9.09,9.07,9.03,9.05,9.07,8.77
2030,2.72,2.72,2.67,2.73,5.60,5.62,5.63,5.53,7.49,7.52,7.45,7.36,7.51,7.47,7.54,7.29
2035,2.36,2.34,2.34,2.36,3.02,2.71,3.15,3.10,6.78,6.72,6.74,6.71,6.72,6.70,6.74,6.72
2040,1.80,1.81,1.78,1.81,2.16,2.56,2.32,2.56,5.14,5.07,5.14,5.02,5.82,5.83,5.85,5.75
2045,1.26,1.26,1.25,1.26,1.93,2.13,2.14,2.19,3.19,3.21,3.23,3.15,4.80,4.78,4.78,4.75
2050,0.94,0.94,0.95,0.95,1.81,1.84,1.81,1.84,2.50,2.53,2.54,2.52,3.72,3.82,3.94,3.86


## 2050 power source mix, against MPP's published 1.5DS

In [4]:
CATEGORY = {"Coal": "Fossil fuel", "Natural Gas": "Fossil fuel",
            "Coal+CCS": "Fossil fuel with capture", "Natural Gas+CCS": "Fossil fuel with capture",
            "Grid": "Grid", "PPA+Grid": "Power purchase agreement",
            "Hydro": "Hydro", "Small Modular Reactor": "Nuclear"}
ORDER = ["Fossil fuel", "Fossil fuel with capture", "Grid",
         "Power purchase agreement", "Hydro", "Nuclear"]

def mix_share_2050(scenario):
    p = production_by_technology(scenario, "smelter")
    p.columns = [CATEGORY[power_source(c)] for c in p.columns]
    m = p.T.groupby(level=0).sum().T
    for c in ORDER:
        if c not in m.columns:
            m[c] = 0.0
    row = m[ORDER].loc[2050]
    return 100 * row / row.sum()

published = pd.read_excel("../../../mpp_aluminium_net_zero_outputs.xlsx",
                          sheet_name="Annual_production_volume_Mt_df")
pub = published[(published.scenario == "1.5DS") & (published.plant_type == "Smelter")
                & (published.year == 2050)].copy()
pub["cat"] = [CATEGORY[power_source(t)] for t in pub.technology]
pub_share = 100 * pub.groupby("cat").value.sum().reindex(ORDER).fillna(0)
pub_share = pub_share / pub_share.sum() * 100

table = pd.DataFrame({"MPP published 1.5DS": pub_share,
                      **{LABELS[s]: mix_share_2050(s) for s in SCENARIOS}}).T.round(1)
table["deviation from published"] = (table - table.loc["MPP published 1.5DS"]).abs().sum(axis=1)
table

,Fossil fuel,Fossil fuel with capture,Grid,Power purchase agreement,Hydro,Nuclear,deviation from published
MPP published 1.5DS,0.00,48.30,9.20,9.40,10.00,23.20,0.00
"SSP1-1.9, none CCS",0.00,1.00,45.70,22.30,6.30,24.80,102.00
"SSP1-1.9, low CCS",0.00,2.60,43.00,22.90,6.20,25.20,98.80
"SSP1-1.9, high CCS",0.00,22.40,34.80,13.30,6.30,23.30,59.20
"SSP1-1.9, unlimited CCS",0.00,48.40,27.30,4.10,6.20,13.90,36.60
"SSP1-2.6, none CCS",5.70,0.00,44.10,39.40,6.20,4.60,141.30
"SSP1-2.6, low CCS",4.90,1.90,51.80,35.20,6.20,0.00,146.70
"SSP1-2.6, high CCS",3.70,38.40,30.10,21.60,6.20,0.00,73.70
"SSP1-2.6, unlimited CCS",1.60,40.20,43.20,8.10,6.20,0.80,71.20
"SSP2-3.7, none CCS",14.00,0.00,35.20,24.10,6.40,20.30,109.50


## Milestone years (smelter anode)

No new investment is the first year after which unabated capacity never rises again. Phase out is the first year at or below 1% of the 2020 level, with 2050 as a backstop (choices 25 to 27). Refinery milestones (digester, calciner) are omitted — refining was not re-run on GCAM demand.

In [5]:
GROUPS = {"anode": (["Carbon Anode"], "smelter", 0)}

def unabated_series(scenario, plant, unabated, part):
    p = production_by_technology(scenario, plant)
    keep = [c for c in p.columns if c.split(" + ")[part] in unabated]
    return p[keep].sum(axis=1)

def milestones(series):
    base = series.loc[2020]
    if base <= 0:
        return None, None
    rises = [y for y in range(2021, 2051) if series[y] > series[y - 1] + 1e-9]
    no_new = (max(rises) + 1) if rises else 2021
    below = [y for y in range(2020, 2051) if series[y] <= 0.01 * base]
    return no_new, (below[0] if below else 2050)

rows = []
for s in SCENARIOS:
    for name, (un, plant, part) in GROUPS.items():
        nn, po = milestones(unabated_series(s, plant, un, part))
        rows.append({"Scenario": LABELS[s], "Asset group": name,
                     "No new investment": nn, "Phase out": po})
pd.DataFrame(rows).pivot(index="Asset group", columns="Scenario",
                         values=["No new investment", "Phase out"])

No new investment                                                                                                                                                                                    ...          Phase out  \
Scenario    SSP1-1.9, high CCS SSP1-1.9, low CCS SSP1-1.9, none CCS SSP1-1.9, unlimited CCS SSP1-2.6, high CCS SSP1-2.6, low CCS SSP1-2.6, none CCS SSP1-2.6, unlimited CCS SSP2-3.7, high CCS SSP2-3.7, low CCS  ... SSP1-2.6, none CCS   
Asset group                                                                                                                                                                                                       ...                      
anode                     2031              2031               2031                    2031               2031              2031               2031                    2031               2031              2031  ...               2050   

                                                                                                                                                                                                     
Scenario    SSP1-2.6, unlimited CCS SSP2-3.7, high CCS SSP2-3.7, low CCS SSP2-3.7, none CCS SSP2-3.7, unlimited CCS SSP2-4.5, high CCS SSP2-4.5, low CCS SSP2-4.5, none CCS SSP2-4.5, unlimited CCS  
Asset group                                                                                                                                                                                          
anode                          2050               2050              2050               2050                    2050               2050              2050               2050                    2050  

[1 rows x 32 columns]